# Setup and Connect to MilVus Vector Database

In [27]:
from pymilvus import connections, db, CollectionSchema, FieldSchema, DataType, Collection, utility
from uuid import uuid4

## Connect to Milvus

In [4]:
mv_conn = connections.connect(
  alias="default",
  host='172.29.0.6',
  port='19530'
)

## Disconnect from Milvus

In [ ]:
connections.disconnect("default")

## Create Database

In [10]:
db.list_database()

['default', 'ring_cam']

## Create Database in Milvus

In [9]:
database = db.create_database("ring_cam")

In [8]:
# db.drop_database("book")

## Create Field Schema and Collection

1. Collection for each reference images set
2. All Images in the dataset

In [21]:
vector_uuid = FieldSchema(
  name="uuid",
  dtype=DataType.VARCHAR,
  max_length=200,
  default_value=uuid4().hex,
  is_primary=True,
)
vector_feature = FieldSchema(
  name="embedding",
  dtype=DataType.FLOAT_VECTOR,
  dim=576
)
schema = CollectionSchema(
  fields=[vector_uuid, vector_feature],
  description="Vector and UUID for query",
  enable_dynamic_field=True
)

In [22]:
collection_name_crops = "all_crops"
collection_name_ref01 = "reference_01"

In [25]:
collection = Collection(
    name=collection_name_crops,
    schema=schema,
    using='default',
    shards_num=2
    )

In [30]:
all_crops = Collection("all_crops") 

## Insert Data

In [41]:
data = [[uuid4().hex], [[float(i) for i in range(576)]]]
# must be in list format, embedding is list of lists

In [42]:
all_crops.insert(data)

(insert count: 1, delete count: 0, upsert count: 0, timestamp: 444375658735599619, success count: 1, err count: 0)

## Prepare Index

In [63]:
index_params = {
  "metric_type":"L2",
  "index_type":"IVF_FLAT",
  "params":{"nlist": 16384}
}

In [45]:
# prepare index
all_crops.create_index(field_name="embedding", index_params=index_params)

Status(code=0, message=)

In [46]:
utility.index_building_progress("all_crops")

{'total_rows': 0, 'indexed_rows': 0, 'pending_index_rows': 0}

## Search Milvus

In [47]:
all_crops.load()

In [55]:
search_params = {
    "metric_type": "L2",
    'index_type':"IVF_FLAT", 
    "params": {'nlist': 16384}
}

In [56]:
results = collection.search(
    data=[[float(i)+1.2 for i in range(576)]], 
    anns_field="embedding", 
    # the sum of `offset` in `param` and `limit` 
    # should be less than 16384.
    param=search_params,
    limit=10,
    expr=None,
    # set the names of the fields you want to 
    # retrieve from the search result.
    output_fields=['uuid'],
    consistency_level="Strong"
)

In [62]:
print(results[0].ids)
results[0].distances

['86a6521b64024860b939f5c3f46ddb65', '8048ee3982e54154967c869e701b5426']


[829.44873046875, 829.44873046875]

In [ ]:
# Define a threshold (e.g., similarity >= 0.7)
threshold = 0.7

# Filter results based on the threshold
filtered_results = [result for result in results[0] if result.distance >= threshold]
